<a href="https://colab.research.google.com/github/Lornatt88/maai901-vision-project/blob/main/notebooks/03_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import pandas as pd
import torch
import torch.nn as nn
import kagglehub
from google.colab import drive

path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/maai901_skin"
test_df = pd.read_csv(os.path.join(save_dir, "test.csv"))
classes = ["nv", "bkl", "mel", "bcc"]
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Ready. Test images:", len(test_df), "| Running on:", device)

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Mounted at /content/drive
Ready. Test images: 544 | Running on: cpu


In [4]:
!pip install -q gradio

import gradio as gr
import torch.nn.functional as F
from torchvision import models, transforms

demo_model = models.resnet50()
demo_model.fc = nn.Linear(2048, 4)
demo_model.load_state_dict(torch.load(os.path.join(save_dir, "resnet50_finetuned.pt"), map_location=device))
demo_model.to(device).eval()

demo_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

labels = ["Mole (nv) - benign", "Benign keratosis (bkl) - benign",
          "Melanoma (mel) - malignant", "Basal cell carcinoma (bcc) - malignant"]
T = 0.25

def predict(img):
    x = demo_tf(img.convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        p = F.softmax(demo_model(x), dim=1)[0].cpu().numpy()
    cancer = p[2] + p[3]
    if cancer >= T:
        verdict = f"Possible skin cancer ({cancer:.0%} combined chance). This lesion should be checked by a dermatologist."
    else:
        verdict = f"Likely benign ({1 - cancer:.0%} chance). This is not a diagnosis. See a doctor if you are concerned."
    return {labels[i]: float(p[i]) for i in range(4)}, verdict

examples = [test_df[test_df["dx"] == c].iloc[0]["path"] for c in classes]

gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload a dermatoscopic image"),
    outputs=[gr.Label(num_top_classes=4, label="Probabilities"), gr.Textbox(label="Result")],
    title="Skin Lesion Check (research demo)",
    description="ResNet-50 fine-tuned on HAM10000 (4 classes). For research and education only, not a medical device.",
    examples=examples,
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://24c9f9d3792ec75e51.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
